# Phase 3 — GDELT Disruption Integration

Pulls live maritime-relevant disruption events from the GDELT GEO 2.0 API, geo-matches them to ports within a buffer radius, computes a severity score, and penalizes affected graph edges.

**Fallback built in:** every successful pull is cached to disk. If the live call fails (network, rate limit, empty response), the notebook automatically falls back to the last cached file so the rest of the pipeline never breaks.

In [66]:
import requests
import pandas as pd
import numpy as np
import networkx as nx
import pickle
import json
import os
from datetime import datetime
from geopy.distance import great_circle

PROCESSED_DIR = "../data/processed"
CACHE_DIR = "../data/processed/gdelt_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# Load the graph and ports from Phase 1/2
with open(f"{PROCESSED_DIR}/maritime_graph.gpickle", "rb") as f:
    G = pickle.load(f)

ports = pd.read_csv(f"{PROCESSED_DIR}/ports_clean.csv")
print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Graph loaded: 1543 nodes, 9462 edges


In [67]:
# --- GDELT GEO 2.0 API query ---
# Docs: https://blog.gdeltproject.org/gdelt-geo-2-0-api-debuts/
# Returns GeoJSON with article-level events including lat/lon

GDELT_GEO_URL = "https://api.gdeltproject.org/api/v2/geo/geo"

# Maritime/disruption-relevant keywords
MARITIME_QUERY = (
    '("port closure" OR "canal blockage" OR "shipping disruption" OR '
    '"maritime attack" OR "vessel attack" OR "port strike" OR "canal drought" OR '
    '"red sea" OR "suez canal" OR "panama canal" OR "strait of hormuz")'
)

def fetch_gdelt_events(query=MARITIME_QUERY, timespan="7d", max_records=250):
    params = {
        "query": query,
        "mode": "pointdata",   # <-- ADD THIS: required for point-level geojson with lat/lon
        "format": "geojson",
        "timespan": timespan,
    }
    response = requests.get(GDELT_GEO_URL, params=params, timeout=20)
    response.raise_for_status()
    data = response.json()

    events = []
    for feature in data.get("features", []):
        geom = feature.get("geometry", {})
        props = feature.get("properties", {})
        coords = geom.get("coordinates", None)
        if coords is None or len(coords) != 2:
            continue
        lon, lat = coords
        events.append({
            "latitude": lat,
            "longitude": lon,
            "name": props.get("name", ""),
            "count": props.get("count", 1),
            "tone": props.get("meanavgtone", 0.0) if "meanavgtone" in props else props.get("tone", 0.0),
            "shareimage": props.get("shareimage", ""),
            "html": props.get("html", ""),
        })
    return events

In [68]:
# --- Fetch with cache fallback ---

def get_events_with_fallback():
    timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    cache_path = f"{CACHE_DIR}/gdelt_events_{timestamp}.json"
    latest_pointer = f"{CACHE_DIR}/latest.json"

    try:
        events = fetch_gdelt_events()
        if not events:
            raise ValueError("GDELT returned zero events")

        # Save this successful pull as the new cache
        with open(cache_path, "w") as f:
            json.dump(events, f)
        with open(latest_pointer, "w") as f:
            json.dump(events, f)

        print(f"Live GDELT pull succeeded: {len(events)} events")
        return events, "live"

    except Exception as e:
        print(f"Live GDELT pull failed ({e}). Falling back to cache...")
        if os.path.exists(latest_pointer):
            with open(latest_pointer, "r") as f:
                events = json.load(f)
            print(f"Loaded {len(events)} events from cache.")
            return events, "cached"
        else:
            print("No cache available. Using hardcoded fallback scenarios instead.")
            fallback_events = [
                {"latitude": 30.5, "longitude": 32.3, "name": "Suez Canal disruption (fallback)", "count": 10, "tone": -8.0},
                {"latitude": 13.0, "longitude": 43.0, "name": "Red Sea disruption (fallback)", "count": 10, "tone": -9.0},
                {"latitude": 9.1, "longitude": -79.7, "name": "Panama Canal disruption (fallback)", "count": 8, "tone": -5.0},
            ]
            return fallback_events, "hardcoded_fallback"

events, source = get_events_with_fallback()
print(f"Source: {source} | Total events: {len(events)}")

C:\Users\varch\AppData\Local\Temp\ipykernel_9636\1474109022.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


Live GDELT pull failed (404 Client Error: Not Found for url: https://api.gdeltproject.org/api/v2/geo/geo?query=%28%22port+closure%22+OR+%22canal+blockage%22+OR+%22shipping+disruption%22+OR+%22maritime+attack%22+OR+%22vessel+attack%22+OR+%22port+strike%22+OR+%22canal+drought%22+OR+%22red+sea%22+OR+%22suez+canal%22+OR+%22panama+canal%22+OR+%22strait+of+hormuz%22%29&mode=pointdata&format=geojson&timespan=7d). Falling back to cache...
No cache available. Using hardcoded fallback scenarios instead.
Source: hardcoded_fallback | Total events: 3


In [69]:
events_df = pd.DataFrame(events)
print(events_df.shape)
events_df.head(10)

(3, 5)


,latitude,longitude,name,count,tone
0,30.5,32.3,Suez Canal disruption (fallback),10,-8.0
1,13.0,43.0,Red Sea disruption (fallback),10,-9.0
2,9.1,-79.7,Panama Canal disruption (fallback),8,-5.0


## Geo-match events to ports
For each port, find events within a buffer radius (nautical miles). If any events fall within the buffer, compute a severity score for that port from event count + tone.

In [70]:
BUFFER_NM = 250  # matches the paper's ~200km buffer, converted to nm and rounded

def compute_port_severity(port_lat, port_lon, events_df, buffer_nm=BUFFER_NM):
    """
    Returns a severity score in [0, 1] for a port based on nearby GDELT events.
    Severity combines event count (more events = more severe) and tone
    (more negative tone = more severe), normalized to [0,1].
    """
    if events_df.empty:
        return 0.0

    port_coord = (port_lat, port_lon)
    nearby = []
    for _, ev in events_df.iterrows():
        try:
            dist = great_circle(port_coord, (ev['latitude'], ev['longitude'])).nautical
        except Exception:
            continue
        if dist <= buffer_nm:
            nearby.append(ev)

    if not nearby:
        return 0.0

    nearby_df = pd.DataFrame(nearby)
    total_count = nearby_df['count'].sum()
    avg_tone = nearby_df['tone'].mean()  # typically negative for bad news, e.g. -10 to -5

    # Normalize: count contributes via log-scale (diminishing returns), tone via clipped negative range
    count_component = min(np.log1p(total_count) / np.log1p(50), 1.0)  # saturate around 50 mentions
    tone_component = min(max(-avg_tone / 10.0, 0.0), 1.0)  # -10 tone -> 1.0, 0 tone -> 0.0

    severity = 0.5 * count_component + 0.5 * tone_component
    return round(float(severity), 3)

In [71]:
# Compute severity for every port in the graph
port_severity = {}
for node_id, attrs in G.nodes(data=True):
    sev = compute_port_severity(attrs['latitude'], attrs['longitude'], events_df)
    port_severity[node_id] = sev
    if sev > 0:
        nx.set_node_attributes(G, {node_id: sev}, name='disruption_severity')

# Ports with zero severity still get the attribute, defaulted to 0.0
nx.set_node_attributes(G, {n: port_severity.get(n, 0.0) for n in G.nodes()}, name='disruption_severity')

affected_ports = {k: v for k, v in port_severity.items() if v > 0}
print(f"Ports affected by disruption: {len(affected_ports)} / {G.number_of_nodes()}")
print("Top 10 most severe:")
for port_id, sev in sorted(affected_ports.items(), key=lambda x: -x[1])[:10]:
    print(f"  {G.nodes[port_id]['port_name']} ({G.nodes[port_id]['country']}): {sev}")

Ports affected by disruption: 35 / 1543
Top 10 most severe:
  Berbera (Somalia): 0.755
  Al Mukha (Yemen): 0.755
  Ras Isa Marine Terminal (Yemen): 0.755
  Assab (Eritrea): 0.755
  Jizan (Saudi Arabia): 0.755
  Al Ahmadi (Yemen): 0.755
  Aden (Yemen): 0.755
  Djibouti (Djibouti): 0.755
  Doraleh (Djibouti): 0.755
  Elat (Israel): 0.705


## Penalize edges touching disrupted ports
An edge's `disrupted_weight` = `base_weight * (1 + penalty_multiplier * max(severity of the two endpoint ports))`.
This keeps `base_weight` (pure distance) untouched — Baseline 1 can use either depending on whether disruption should apply.

In [72]:
PENALTY_MULTIPLIER = 5.0  # a fully-severe (1.0) endpoint multiplies edge cost 6x

for u, v, data in G.edges(data=True):
    sev_u = G.nodes[u].get('disruption_severity', 0.0)
    sev_v = G.nodes[v].get('disruption_severity', 0.0)
    max_sev = max(sev_u, sev_v)

    penalty_factor = 1.0 + PENALTY_MULTIPLIER * max_sev
    data['disrupted_weight'] = data['base_weight'] * penalty_factor
    data['disruption_severity_edge'] = max_sev

penalized_edges = [(u, v) for u, v, d in G.edges(data=True) if d['disruption_severity_edge'] > 0]
print(f"Edges penalized: {len(penalized_edges)} / {G.number_of_edges()}")

Edges penalized: 292 / 9462


In [73]:
# --- Save the disruption-tagged graph (separate file — keeps Phase 2's clean graph untouched) ---

output_path = f"{PROCESSED_DIR}/maritime_graph_disrupted.gpickle"
with open(output_path, "wb") as f:
    pickle.dump(G, f)

print(f"Disruption-tagged graph saved to {output_path}")
print(f"Data source used: {source}")
print(f"Affected ports: {len(affected_ports)}, Penalized edges: {len(penalized_edges)}")

Disruption-tagged graph saved to ../data/processed/maritime_graph_disrupted.gpickle
Data source used: hardcoded_fallback
Affected ports: 35, Penalized edges: 292


diagnosis

In [74]:
import requests

try:
    resp = requests.get(
        "https://api.gdeltproject.org/api/v2/geo/geo",
        params={
            "query": '"red sea" OR "suez canal" OR "port closure"',
            "format": "geojson",
            "timespan": "7d",
        },
        timeout=20
    )
    print("Status code:", resp.status_code)
    print("Response text (first 500 chars):", resp.text[:500])
except Exception as e:
    print("Exception type:", type(e).__name__)
    print("Exception message:", e)

Status code: 404
Response text (first 500 chars): <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01//EN" "http://www.w3.org/TR/html4/strict.dtd">
<html><head>
<title>404 Not Found</title>
</head><body>
<h1>Not Found</h1>
<p>The requested URL was not found on this server.</p>
</body></html>

